# 📊 Task 3: Data Visualization
### CodeAlpha Data Analytics Internship
**Dataset:** Netflix Movies & TV Shows  
**Source:** Built-in generation (no download needed) + Seaborn datasets  
**Goal:** Transform raw data into compelling charts that tell a clear data story.

In [ ]:
# ─────────────────────────────────────────────
# STEP 1: Setup & Generate Rich Sample Dataset
# ────────────────────────────────────────────

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

# --- Generate a realistic Netflix-style dataset ---
n = 1000

content_types = np.random.choice(['Movie','TV Show'], n, p=[0.7, 0.3])
countries = np.random.choice(
    ['United States','India','United Kingdom','Japan','South Korea','France','Germany','Canada','Spain','Brazil'],
    n, p=[0.35,0.20,0.10,0.08,0.07,0.05,0.04,0.04,0.04,0.03]
)
years = np.random.choice(range(2010, 2023), n, 
                          p=[0.02,0.02,0.03,0.04,0.05,0.07,0.09,0.11,0.13,0.14,0.14,0.12,0.04])
ratings = np.random.choice(['TV-MA','TV-14','TV-PG','R','PG-13','PG','G','NR'], n,
                            p=[0.30,0.25,0.15,0.10,0.08,0.05,0.03,0.04])

genres_list = ['Drama','Comedy','Action & Adventure','Documentary','Thriller',
               'Romance','Horror','Sci-Fi','Kids','Anime','Crime','Biography']
genres = np.random.choice(genres_list, n)

duration_movies = np.random.normal(100, 25, n).clip(60, 180).astype(int)
duration_shows  = np.random.choice([1,2,3,4,5,6,7,8], n, p=[0.25,0.20,0.20,0.15,0.10,0.05,0.03,0.02])

df = pd.DataFrame({
    'type': content_types,
    'country': countries,
    'year_added': years,
    'rating': ratings,
    'genre': genres,
    'duration_minutes': [duration_movies[i] if content_types[i]=='Movie' else duration_shows[i]*45 for i in range(n)]
})

print("✅ Dataset generated!")
print(f"Shape: {df.shape}")
df.head(8)

In [ ]:
# ─────────────────────────────────────────────
# STEP 2: DASHBOARD — Overview Charts
# ─────────────────────────────────────────────

fig = plt.figure(figsize=(22, 14))
fig.patch.set_facecolor('#0f0f0f')
fig.suptitle('🎬  Netflix Content Dashboard', 
             fontsize=26, fontweight='bold', color='white', y=1.0)

gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

DARK_BG  = '#1a1a2e'
RED      = '#e50914'
GOLD     = '#f5c518'
BLUE     = '#00b4d8'
GREEN    = '#06d6a0'
TEXT     = 'white'
palette  = [RED, BLUE, GREEN, GOLD, '#9b59b6', '#e67e22', '#1abc9c', '#e91e8c']

def style_ax(ax, title):
    ax.set_facecolor(DARK_BG)
    ax.tick_params(colors=TEXT, labelsize=10)
    ax.set_title(title, fontsize=13, fontweight='bold', color=TEXT, pad=10)
    for spine in ax.spines.values():
        spine.set_edgecolor('#333')
    ax.xaxis.label.set_color(TEXT)
    ax.yaxis.label.set_color(TEXT)

# ── Chart 1: Movies vs TV Shows (Donut) ──
ax1 = fig.add_subplot(gs[0, 0])
ax1.set_facecolor(DARK_BG)
counts = df['type'].value_counts()
wedges, texts, autotexts = ax1.pie(
    counts, labels=counts.index, autopct='%1.1f%%',
    colors=[RED, BLUE], startangle=90,
    wedgeprops={'edgecolor':'#0f0f0f','linewidth':3,'width':0.65},
    textprops={'color': TEXT, 'fontsize': 12})
for at in autotexts: at.set_fontsize(13); at.set_fontweight('bold'); at.set_color('white')
ax1.set_title('Movies vs TV Shows', fontsize=13, fontweight='bold', color=TEXT)

# ── Chart 2: Top Countries ──
ax2 = fig.add_subplot(gs[0, 1])
top_countries = df['country'].value_counts().head(8)
bars = ax2.barh(top_countries.index[::-1], top_countries.values[::-1],
                color=[RED if i==0 else BLUE for i in range(len(top_countries))],
                edgecolor='none', height=0.65)
style_ax(ax2, 'Top 8 Countries by Content')
ax2.set_xlabel('Number of Titles', color=TEXT)
for bar in bars:
    ax2.text(bar.get_width()+2, bar.get_y()+bar.get_height()/2,
             str(int(bar.get_width())), va='center', color=TEXT, fontsize=10, fontweight='bold')

# ── Chart 3: Content Added Per Year ──
ax3 = fig.add_subplot(gs[0, 2])
yearly = df.groupby('year_added').size().reset_index(name='count')
ax3.plot(yearly['year_added'], yearly['count'], 
         color=RED, linewidth=2.5, marker='o', markersize=6, markerfacecolor=GOLD)
ax3.fill_between(yearly['year_added'], yearly['count'], alpha=0.2, color=RED)
style_ax(ax3, 'Content Added Per Year')
ax3.set_xlabel('Year')
ax3.set_ylabel('Titles Added')

# ── Chart 4: Genre Distribution ──
ax4 = fig.add_subplot(gs[1, 0])
genre_counts = df['genre'].value_counts().head(10)
bars = ax4.bar(range(len(genre_counts)), genre_counts.values,
               color=palette[:len(genre_counts)], edgecolor='none', width=0.7)
ax4.set_xticks(range(len(genre_counts)))
ax4.set_xticklabels(genre_counts.index, rotation=45, ha='right', fontsize=9)
style_ax(ax4, 'Top 10 Genres')
ax4.set_ylabel('Count')

# ── Chart 5: Content Rating Breakdown ──
ax5 = fig.add_subplot(gs[1, 1])
rating_counts = df['rating'].value_counts()
bars = ax5.bar(rating_counts.index, rating_counts.values,
               color=[GOLD if r in ['G','PG','TV-PG'] else RED for r in rating_counts.index],
               edgecolor='none', width=0.65)
style_ax(ax5, 'Content Rating Distribution')
ax5.set_ylabel('Count')
for bar in bars:
    ax5.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
             str(int(bar.get_height())), ha='center', color=TEXT, fontsize=9, fontweight='bold')

# ── Chart 6: Country vs Genre Heatmap ──
ax6 = fig.add_subplot(gs[1, 2])
top5_countries = df['country'].value_counts().head(5).index
top5_genres    = df['genre'].value_counts().head(6).index
pivot = df[df['country'].isin(top5_countries) & df['genre'].isin(top5_genres)]\
          .groupby(['country','genre']).size().unstack(fill_value=0)
sns.heatmap(pivot, ax=ax6, cmap='YlOrRd', annot=True, fmt='d',
            linewidths=0.5, linecolor='#0f0f0f',
            annot_kws={'size':9}, cbar_kws={'shrink':0.8})
ax6.set_title('Country × Genre Heatmap', fontsize=13, fontweight='bold', color=TEXT)
ax6.set_facecolor(DARK_BG)
ax6.tick_params(colors=TEXT, labelsize=9)
ax6.set_xlabel('')
ax6.set_ylabel('')

plt.tight_layout()
plt.savefig('Task3_DataVisualization/netflix_dashboard.png', dpi=150, 
            bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print("✅ Saved: netflix_dashboard.png")

In [ ]:
# ─────────────────────────────────────────────
# STEP 3: Deep Dive — Genre & Duration Analysis
# ─────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Genre & Duration Deep Dive', fontsize=18, fontweight='bold')

# Grouped bar: Movies vs TV Shows per Genre
ax = axes[0]
movies = df[df['type']=='Movie']['genre'].value_counts()
shows  = df[df['type']=='TV Show']['genre'].value_counts()
all_genres = movies.index[:10]
x = range(len(all_genres))
width = 0.4
ax.bar([i - width/2 for i in x], [movies.get(g, 0) for g in all_genres], 
       width=width, label='Movie', color='#e50914', edgecolor='white', linewidth=0.5)
ax.bar([i + width/2 for i in x], [shows.get(g, 0) for g in all_genres],
       width=width, label='TV Show', color='#00b4d8', edgecolor='white', linewidth=0.5)
ax.set_xticks(list(x))
ax.set_xticklabels(all_genres, rotation=40, ha='right')
ax.set_title('Genre Breakdown: Movies vs TV Shows', fontsize=14, fontweight='bold')
ax.set_ylabel('Count')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Duration Boxplot by Type
ax2 = axes[1]
movies_dur = df[df['type']=='Movie']['duration_minutes']
shows_dur  = df[df['type']=='TV Show']['duration_minutes']
bp = ax2.boxplot([movies_dur, shows_dur], labels=['Movies\n(minutes)', 'TV Shows\n(minutes×eps)'],
                  patch_artist=True, notch=True,
                  boxprops=dict(facecolor='#e5091466', color='#e50914'),
                  medianprops=dict(color='white', linewidth=2),
                  whiskerprops=dict(color='#aaa'),
                  capprops=dict(color='#aaa'),
                  flierprops=dict(marker='o', color='#aaa', markersize=3))
bp['boxes'][1].set_facecolor('#00b4d866')
bp['boxes'][1].set_edgecolor('#00b4d8')
ax2.set_title('Duration Distribution by Content Type', fontsize=14, fontweight='bold')
ax2.set_ylabel('Duration (minutes)')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('Task3_DataVisualization/genre_duration_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: genre_duration_analysis.png")

In [ ]:
# ─────────────────────────────────────────────
# STEP 4: Trend Analysis — Animated-Style Line Chart
# ─────────────────────────────────────────────

plt.figure(figsize=(16, 7))
plt.style.use('seaborn-v0_8-whitegrid')

top_genres = df['genre'].value_counts().head(5).index
colors = ['#e50914','#00b4d8','#06d6a0','#f5c518','#9b59b6']

for i, genre in enumerate(top_genres):
    genre_year = df[df['genre']==genre].groupby('year_added').size().reset_index(name='count')
    plt.plot(genre_year['year_added'], genre_year['count'],
             marker='o', linewidth=2.5, color=colors[i],
             label=genre, markersize=7, markerfacecolor='white',
             markeredgewidth=2, markeredgecolor=colors[i])

plt.title('📈 Genre Trends Over the Years (Top 5 Genres)', fontsize=17, fontweight='bold')
plt.xlabel('Year', fontsize=13)
plt.ylabel('Number of Titles Added', fontsize=13)
plt.legend(fontsize=11, loc='upper left')
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('Task3_DataVisualization/genre_trends.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: genre_trends.png")

## ✅ Task 3 Summary

| Visual | Insight |
|--------|---------|
| Donut Chart | 70% Movies, 30% TV Shows |
| Top Countries | USA & India dominate content |
| Yearly Trend | Rapid growth from 2015 onwards |
| Genre Heatmap | Drama & Comedy most popular across all countries |
| Duration | Movies avg ~100 min; TV shows vary widely |

**Skills demonstrated:** Multi-chart dashboards, color storytelling, Seaborn heatmaps, trend analysis, custom dark theme design.